### Pytorch Fundamentals 
Major Python based deep learning library really good for research and prototyping
## Important components

-  Tensor library for efficient computing (extension of numpy + feature which accelerate computation on GPUs)
-  Include utilities for automatic differentiation engine - Automatic computation of gradients for tensor operations >> simplify backpropogation optimization
-  Deep learning library - provides modular, flexibile and efficient building blocks, pretrained models, loss functions and optimizers

In [1]:
import torch

In [2]:
print(torch.backends.mps.is_available())

True


In [3]:
# Create tensors similiar to numpy arrays
tensor0d = torch.tensor(1) #1
tensor1d = torch.tensor([1, 2, 3]) #2
tensor2d = torch.tensor([[1, 2],[3, 4]]) #3
tensor3d = torch.tensor([[[1, 2], [3, 4]],[[5, 6], [7, 8]]])
print(tensor3d.shape)

torch.Size([2, 2, 2])


In [4]:
# multiplication
print(tensor2d.matmul(tensor2d.T))

tensor([[ 5, 11],
        [11, 25]])


In [5]:
print(tensor2d @ tensor2d.T)

tensor([[ 5, 11],
        [11, 25]])


In [6]:
# Pytorch autograd system
# Compute gradients in the dynamic compuatational graph
# Pytorch automatically builds the computation graph in the background by tracking computations of tensors.
# A simple logistic regression forward pass
# Chain rule is used to calculate gradients, basically partial derivative of multivariate loss function wrt parameters

import torch.nn.functional as F #1
y = torch.tensor([1.0]) #2
x1 = torch.tensor([1.1]) #3
w1 = torch.tensor([2.2]) #4
b = torch.tensor([0.0]) #5
z = x1 * w1 + b #6
a = torch.sigmoid(z) #7
loss = F.binary_cross_entropy(a, y)
print(loss)

tensor(0.0852)


In [18]:
# Set require_grad = True for parameters to compute the gradients
import torch.nn.functional as F
from torch.autograd import grad
y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)
z = x1 * w1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)

# Pytorch destroy computational graph after calculating the gradients
# save retain_graph = True if the the graphs are needed later.
grad_L_w1 = grad(loss, w1, retain_graph=True) #
grad_L_b = grad(loss, b, retain_graph=True)

print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [14]:
# .grad will gives the gradient values asssociated
print(w1.grad)

None


In [19]:
## final_fuction.backward() will calculate the gradient at all the edges of the compuational graph automatically
## will be stored for each tensor via their .grad attribures
loss.backward()
print(w1.grad, b.grad)

tensor([-0.0898]) tensor([-0.0817])


###  Important points
1. Each node calculate their local derivatives during the forward prop. Later then uses chain rule to calculate the gradients. It does not matter when the tensor is scalar, vector or mult dimensional matrix.
2. A parameter gradient is simply the derivative of the scalar loss wrt to the each element of the weight tensor.
3. Ideally you want to calculate the derivative of each scalar loss wrt to each parameter. This usually result in a Jacobian matrix. If you  have x element R_m and y element of R_n. This will result in big m * n matrix. Storing them is not feasible.
4. Instead backpropogation can be thought as a vector jacobian product.
5. At each node, you calculate ***incoming_gradient * local Jacobian product***. The backpropogation is basically sequence of such calculations to replicate the whole Jacobian. 

In [47]:
## A simple MLP network
## All network components are defined in the init constructor which subclasses the Module
## Forward method define the interaction between network components

class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()
        self.layers = torch.nn.Sequential(
        # 1st hidden layer
        torch.nn.Linear(num_inputs, 30), #
        torch.nn.ReLU(), #3
        # 2nd hidden layer
        torch.nn.Linear(30, 20), #4
        torch.nn.ReLU(),
        # output layer
        torch.nn.Linear(20, num_outputs),
        )
    def forward(self, x): # This is where the data is passed into the forward function, where the data flow begins
        logits = self.layers(x)
        return logits #5

In [40]:
# Call the model
model = NeuralNetwork(50, 5)
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=5, bias=True)
  )
)


In [26]:
num_params = sum(p.numel() for p in model.parameters())
print(f"Total number of trainable model parameters : {num_params}")

Total number of trainable model parameters : 2255


In [27]:
print(model.layers[0].weight)

Parameter containing:
tensor([[-0.1141,  0.0944,  0.1208,  ...,  0.0974, -0.0063,  0.0160],
        [-0.1183, -0.0034,  0.1129,  ...,  0.0798, -0.0749,  0.0426],
        [ 0.0561,  0.0785, -0.0394,  ..., -0.0225,  0.0446, -0.1308],
        ...,
        [-0.0652,  0.1402,  0.0267,  ...,  0.0956, -0.0795, -0.0031],
        [ 0.1081,  0.0711,  0.0942,  ...,  0.0845, -0.0680,  0.1160],
        [-0.1412,  0.0785,  0.0043,  ...,  0.0849,  0.1160,  0.0950]],
       requires_grad=True)


In [28]:
print(model.layers[0].weight.shape)

torch.Size([30, 50])


In [41]:
# create a toy dataset and define how data is loaded using data loader
X_train = torch.tensor([
[-1.2, 3.1],
[-0.9, 2.9],
[-0.5, 2.6],
[2.3, -1.1],
[2.7, -1.5]
])
y_train = torch.tensor([0, 0, 0, 1, 1])
X_test = torch.tensor([
[-0.8, 2.8],
[2.6, -1.6],
])
y_test = torch.tensor([0, 1])

In [42]:
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index): #1 define how to get a single training instance
        one_x = self.features[index] #1
        one_y = self.labels[index] #1
        return one_x, one_y #1

    def __len__(self): # define the shape of the dataset or tensor shape
        return self.labels.shape[0] #2

# define data in the form of class instances
train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

- Earlier class instances are passed to Dataloader class which does all the shuffling and loading data in batches
- We need to paralleize the loading process here, so that the model won't sit idle until the next batch is loaded
  Otherwise it will create the loading bottleneck where model wait for the next batch to be loaded with mutliple  workers, the data loader can prepare the next batches of data ready to be loaded in the background and queue it up and so no wating
- Mostly relevant for big datasets.
- This process essentially parallelize the preparation of batches and loading them into the CPU memory. Now they need to be moved from CPU to GPU memory for further model training.
- This way when the GPU finishes training, it does not need to wait for CPU to prepare the next batch as it is already prepared in the background and loaded into CPU memory

In [43]:
from torch.utils.data import DataLoader

torch.manual_seed(123)

train_loader = DataLoader(
dataset=train_ds, #1
batch_size=2, # batch size
shuffle=True, #2 shuffle the data after each iteration, randomize the process and remove repetitive gradient update cycles
num_workers=0, #3 parallelize the loading process in GPU (remain idle during cpu load the data), mulitple batches can be trained and updated simultaneously
drop_last=True # Let'say the batch size =32 and last batch has only 5 elements, it can disturb convergence, we can drop it here.
)

test_loader = DataLoader(
dataset=test_ds,
batch_size=2,
shuffle=False, #4
num_workers=0,
drop_last=True
)

In [44]:
# we can iterate over the dataset now
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])


- mode.train() and model.eval() are used for training and evaluation respectiveley
- Necessary for components which behaves differently during training and evaluation
- eg. drop out is completely turned off in evaluation and batch normalization
- drop out freezes certain neural outputs to make neurons learn independent and robust features. A drop out scaling is carried out to make up for the loss in signal levels.
- In batch normalization, each input feature unit is normalized across batch samples. During evaluation, it is turned off and utilized the mean and variance from the training. 

In [50]:
# Modeling process starts here

torch.manual_seed(123)
# Calling the model instance here
model = NeuralNetwork(num_inputs=2, num_outputs=2)    #1

# Define the optimization process for updating the weights
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)            # Calling stochastic gradient descent optimizer

num_epochs = 3
for epoch in range(num_epochs): 

    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features) # Forward propogation

        loss = F.cross_entropy(logits, labels) # Calculate the log loss

        optimizer.zero_grad()   # Set the gradients from previous batch to zero as it is an accumulative quantity
        loss.backward()         # Backpropogation and compute the gradients of loss wrt to model parameters, basically populate the compuational graph with gradients
        optimizer.step()        # Optimizer updates the model parameters using the calculated gradients

        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train Loss: {loss:.2f}")

    model.eval() # usually used to evaluate the validation set and calculate the loss at each iteration.
    with torch.no_grad(): # turn of the computational graph and gradient calculation
        outputs = model(X_train)
    print(outputs)  
        
torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1) # conducted seperately for efficiency and numerical stability
print(probas)
   
predictions = torch.argmax(probas, dim=1)
print(predictions)

Epoch: 001/003 | Batch 000/002 | Train Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train Loss: 0.65
tensor([[ 1.0761, -1.9320],
        [ 0.9304, -1.7248],
        [ 0.7272, -1.4390],
        [-0.2722,  0.2662],
        [-0.3136,  0.3228]])
Epoch: 002/003 | Batch 000/002 | Train Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train Loss: 0.13
tensor([[ 2.7292, -4.0033],
        [ 2.4228, -3.6111],
        [ 1.9964, -3.0593],
        [-1.2539,  1.2540],
        [-1.4510,  1.4673]])
Epoch: 003/003 | Batch 000/002 | Train Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train Loss: 0.00
tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])
tensor([[0.9991, 0.0009],
        [0.9982, 0.0018],
        [0.9949, 0.0051],
        [0.0491, 0.9509],
        [0.0307, 0.9693]])
tensor([0, 0, 0, 1, 1])


- Different hyperparameters here could be number of hidden layers, learning rate, number of epochs
- Validation dataset is often used for hyperparameter tuning and test set is used to avoid the bias of evaluation set and complete generalization of the model

In [51]:
# Function to calculate the accuracy
def compute_accuracy(model, dataloader):

    model = model.eval()
    correct = 0.0
    total_examples = 0

    for idx, (features, labels) in enumerate(dataloader):

        with torch.no_grad():
            logits = model(features)

        predictions = torch.argmax(logits, dim=1)
        compare = labels == predictions       #1
        correct += torch.sum(compare)      #2
        total_examples += len(compare)

    return (correct / total_examples).item()    #3

print(compute_accuracy(model, train_loader))  
print(compute_accuracy(model, test_loader))  

1.0
1.0


### GPU training part

In [52]:
print(torch.cuda.is_available())

False


In [53]:
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)  

In [54]:
print(device)

mps


In [55]:
tensor_1 = torch.tensor([1., 2., 3.])
tensor_2 = torch.tensor([4., 5., 6.])
print(tensor_1 + tensor_2)

tensor([5., 7., 9.])


In [56]:
tensor_1 = tensor_1.to(device)
tensor_2 = tensor_2.to(device)
print(tensor_1 + tensor_2)

tensor([5., 7., 9.], device='mps:0')


- Data/tensors need to be on the same device
- Model needs to be copied to the device as well

In [57]:
torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")        #1
model = model.to(device)          #2

optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):

    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
        features, labels = features.to(device), labels.to(device)   #3
        logits = model(features)
        loss = F.cross_entropy(logits, labels) # Loss function

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")

    model.eval()
    # Insert optional model evaluation code”

Epoch: 001/003 | Batch 000/002 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.65
Epoch: 002/003 | Batch 000/002 | Train/Val Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.13
Epoch: 003/003 | Batch 000/002 | Train/Val Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.00
